In [1]:
# vehicle abstraction with PAC guarantee example

# needed libraries
import numpy as np
import scipy.special as sp
import time
from itertools import product
import random
import gurobipy as gp
from gurobipy import GRB
from joblib import Parallel, delayed
from scipy.optimize import fsolve
from math import comb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.stats as stats
from scipy.optimize import brentq
from scipy.stats import truncnorm

In [2]:
# choice of N for repetitve scenario
N = 4100
n_dim = 3 # dimension of state set
N_pos = 700 # N used for computing post in abstraction

In [3]:
# system dynamics
tau = .25 #sampling time
eta_x1, eta_x2 = 0.18, 0.18 #0.18, .18 
eta_x = np.array([eta_x1,eta_x1,eta_x2]) # discretization vector
# set of states
a1, a2, b1, b2, c1, c2 = 0, 10, 0, 10, -np.pi-2*eta_x2, np.pi+2*eta_x2
a = np.array([a1, b1, c1])  # lower bounds
b = np.array([a2, b2, c2])  # upper bounds

dim1_hat = np.arange(a1+eta_x1, a2-eta_x1, eta_x1)
dim2_hat = np.arange(b1+eta_x1, b2-eta_x1, eta_x1)
dim3_hat = np.arange(c1+eta_x2, c2-eta_x2, eta_x2)
X_ha = np.array(list(product(dim1_hat, dim2_hat, dim3_hat))) 
X_haa = np.around(X_ha, 2)

In [4]:
# Sample N i.i.d. pairs (x_i, x_hat_i)
x_samples = np.random.uniform(low=a, high=b, size=(N, 3))
x_hat_indices = np.random.choice(len(X_haa), size=N, replace=True)
x_hat_samples = X_haa[x_hat_indices]

# Combine into a list of tuples or array of pairs
X_pairs = list(zip(x_samples, x_hat_samples))
len(X_pairs)

4100

In [5]:
# system dynamics as black-box simulator
def sys_dyn(x_t, u_t):
    u1, u2 = u_t
    x1, x2, x3 = x_t
    q = np.arctan(np.tan(u2) / 2)
    f_xt1 = x1 + tau*(u1*np.cos(q + x3) / np.cos(q))
    f_xt2 = x2 + tau*(u1*np.sin(q + x3) / np.cos(q))
    f_xt3 = x3 + tau*(u1*np.tan(u2))
    nxt = [max(min(f_xt1, a2), a1), max(min(f_xt2, b2), b1), max(min(f_xt3, c2), c1)]
    nxt = [round(x, 2) for x in nxt]
    return nxt

def generate_grid_centers(a, b, N_pos):
    dim = len(a)
    # Estimate number of points per dimension (evenly)
    k = int(np.round(N_pos ** (1 / dim)))
    total_points = k ** dim

    # Generate grid centers per dimension
    grid_axes = []
    for ai, bi in zip(a, b):
        step = (bi - ai) / k
        centers = ai + (np.arange(k) + 0.5) * step
        grid_axes.append(centers)

    # Cartesian product of all centers (i.e., subgrid centers)
    all_points = np.array(list(product(*grid_axes)))

    # Trim if more than needed
    if total_points > N_pos:
        all_points = all_points[:N_pos]

    return all_points

# for abstract system
def sys_dyn_hat(y, u):
    y = np.asarray(y)
    X_hat_arra = np.array(X_haa)  
    a, b = y-eta_x/2, y+eta_x/2 
    # sampling N points as subgrid centres for each cell to obtain an estimate of the reachable sets as done in the paper
    sampled_points = generate_grid_centers(a, b, N_pos)
    # sampling N iid points from each cell to obtain an estimate of the reachable sets
    # sampled_points = np.random.uniform(a, b, size=(N_pos, len(a)))

    # Compute successors
    successors = np.array([sys_dyn(x, u) for x in sampled_points])

    # Compute mean and max distance
    m = np.mean(successors, axis=0)
    r = np.max(np.abs(successors - m), axis=0)

    # Find points in X_hat_array within the under-approximation of reachable sets
    mask = np.all(np.abs(X_hat_arra - m) <= (r + eta_x / 2), axis=1)

    return X_hat_arra[mask]

In [6]:
# set of inputs
eta_u = .3 
dim_u = np.arange(-1, 1 - eta_u, eta_u)
U_ha = np.array(list(product(dim_u, dim_u)))
U = np.around(U_ha, 2).tolist()
M = len(U)
U_array = np.array(U)
print(M)

ubpr = 100
lbp, ubp = -200, 100
lbpc, ubpc = -200, 150
lbe, ube = -8.2, 0 

36


#### the SOP

In [7]:
m1 = gp.Model("PAC_ASF")

vartheta = m1.addVar(vtype = GRB.CONTINUOUS, name = "vartheta", lb = lbe, ub = ube)

eps_vars = 1/len(X_pairs) 
# eps_vars = {l:m1.addVar(vtype = GRB.CONTINUOUS, name = f"eps_{l}", lb = 0, ub = ubpr) for l in range(len(X_pairs))}
# sum_eps = gp.quicksum(eps_var for eps_var in eps_vars)
# m1.addConstr(sum_eps == 1)

# eps_varsp = {l:m1.addVar(vtype = GRB.CONTINUOUS, name = f"epsp_{l}", lb = 0, ub = ubpr) for l in range(len(X_pairs))}
# sum_epsp = gp.quicksum(eps_var for eps_var in eps_varsp)
# m1.addConstr(sum_epsp == 1)
m1.update()

Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11


In [8]:
# gamma_b = 10 
# delta_b = gamma_b * np.linalg.norm(np.array(eta_x), ord=np.inf) **2 # ||eta_x||^2 * gamma
gamma_b = m1.addVar(vtype = GRB.CONTINUOUS, name = "gamma_b", lb = 0.0001, ub = ubpr)
delta_b = m1.addVar(vtype = GRB.CONTINUOUS, name = "delta_b", lb = 0.0001, ub = ubpr)
tau_b = 2#m1.addVar(vtype = GRB.CONTINUOUS, name = "tau_b", lb = 0, ub = ubpr) # this is \rho in the paper

num_variables = 28  # number of coefficients as a result of the chosen degree of asf
variable_names = [f"lambda_{i}" for i in range(1, num_variables + 1)]
variable_objs = {name_i: m1.addVar(vtype=GRB.CONTINUOUS, name=name_i, lb=lbpc, ub=ubpc) for name_i in variable_names}

def Asf(x, x_hat):
    # Coefficients from the declared variables
    c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15,c16,c17,c18,c19,c20,c21,c22,c23,c24,c25,c26,c27,c28 = variable_objs.values()
    x1, x2, x3, y1, y2, y3 = x[0], x[1], x[2], x_hat[0], x_hat[1], x_hat[2]
    # Polynomial expression using the variables
    result = (
        c1 + c2*x1 + c3*x2 + c4*x3 + c5*y1 + c6*y2 + c7*y3 + 
        c8*x1**2 + c9*x1*x2 + c10*x1*x3 + c11*x1*y1 + c12*x1*y2 + c13*x1*y3 +
        c14*x2**2 + c15*x2*x3 + c16*x2*y1 + c17*x2*y2 + c18*x2*y3 +
        c19*x3**2 + c20*x3*y1 + c21*x3*y2 + c22*x3*y3 +
        c23*y1**2 + c24*y1*y2 + c25*y1*y3 +
        c26*y2**2 + c27*y2*y3 +
        c28*y3**2
    )
    return result

m1.update()

In [9]:
# Define lambda functions for efficiency
max_abs_diff = lambda x, y: gamma_b * np.max(np.abs(x - y))**2
asf_value = lambda x, y: Asf(x, y)

# First ASF condition
t_init = time.time()
# sum1 = gp.quicksum(eps_vars[l] * (max_abs_diff(x, y) - asf_value(x, y)) for l, (x,y) in enumerate(X_pairs))
sum1 = gp.quicksum(eps_vars * (max_abs_diff(x, y) - asf_value(x, y)) for (x,y) in X_pairs)
m1.addConstr(sum1 <= vartheta)
t_fin = time.time()
diff_t = t_fin - t_init
print(diff_t)
m1.update()

0.21984219551086426


In [10]:
t_init = time.time()
# ASF 2 Enforce sum2 - Asf <= vartheta

X0 = [X_pairs[l][0] for l in range(len(X_pairs))]
X1 = [X_pairs[l][1] for l in range(len(X_pairs))]
U_list = list(U)

# Only parallelize the pure numerical dynamics (no Gurobi objects)
def compute_dynamics(x0, x1, u):
    dyn_hat_out = sys_dyn_hat(x1, u)  
    dyn_out     = sys_dyn(x0, u)      
    return dyn_out, list(dyn_hat_out)

dyn_results = Parallel(n_jobs=-1, backend="threading", verbose=2)(
    delayed(compute_dynamics)(X0[l], X1[l], U_list[i])
    for l in range(len(X_pairs))
    for i in range(len(U_list))
)

# All Gurobi work (asf_value, addConstr) stays sequential in main process
m1.addConstrs(
    (
        gp.quicksum(
            asf_value(dyn_results[l * len(U_list) + i][0], yn) / len(dyn_results[l * len(U_list) + i][1]) # sigma taken uniformly
            for yn in dyn_results[l * len(U_list) + i][1]
        )
        - tau_b * asf_value(X0[l], X1[l]) + (tau_b - 1) * delta_b <= vartheta
        for l in range(len(X_pairs))
        for i in range(len(U_list))
    ),
    name="asf2"
)

m1.update()
t_fin = time.time()
print(f"Setup time: {t_fin - t_init:.2f}s")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 142 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 345 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 628 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 993 tasks      | elapsed:    7.7s
[Parallel(n_jobs=-1)]: Done 1438 tasks      | elapsed:   11.1s
[Parallel(n_jobs=-1)]: Done 1965 tasks      | elapsed:   15.2s
[Parallel(n_jobs=-1)]: Done 2572 tasks      | elapsed:   20.0s
[Parallel(n_jobs=-1)]: Done 3261 tasks      | elapsed:   25.4s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:   31.4s
[Parallel(n_jobs=-1)]: Done 4881 tasks      | elapsed:   38.0s
[Parallel(n_jobs=-1)]: Done 5812 tasks      | elapsed:   45.5s
[Parallel(n_jobs=-1)]: Done 6825 tasks      | elapsed:   53.4s
[Parallel(n_jobs=-1)]: Done 7918 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 9093 tasks   

Warning for adding constraints: zero or small (< 1e-13) coefficients, ignored
Setup time: 1221.58s


In [11]:
m1.Params.LogToConsole = 0
m1.setParam("NumericFocus", 3)
m1.setParam('NonConvex', 2)

m1.update()
print("Start now")
print("Number of variables:", m1.numVars)
print("Number of constraints:", m1.numConstrs)
t_init = time.time()

m1.setObjective(vartheta, GRB.MINIMIZE)
m1.setParam('Presolve', 0)

m1.optimize()
t_fin = time.time()
diff_t = t_fin - t_init
print("opt_status:", m1.status)

m1.write("vehicle_with_PAC_Infeasible_most_recent.lp")

# Check if optimization was successful
if m1.status == GRB.OPTIMAL:
    with open('vehicle_with_PAC_variables_most_recent.txt', 'w') as file:
        for v in m1.getVars():
            file.write(f'{v.VarName} {v.X}\n')
elif m1.status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m1.computeIIS()
    m1.write("vehicle_with_PAC_Infeasible_most_recent.ilp")
    print("IIS written to vehicle_with_PAC_Infeasible_most_recent.ilp")

    with open("vehicle_with_PAC_Infeasible_most_recent.txt", "w") as f:
        for c in m1.getConstrs():
            if c.IISConstr:
                f.write(f"Infeasible constraint: {c.ConstrName}\n")
        for v in m1.getVars():
            if v.IISLB or v.IISUB:
                f.write(f"Infeasible bound on variable: {v.VarName}\n")

else:
    print("Optimization was not successful (not optimal or infeasible). Status:", m1.status)

print("Time (in s):", diff_t)

Start now
Number of variables: 31
Number of constraints: 147601
opt_status: 2
Time (in s): 0.36193394660949707


In [12]:
def prune_constraints_inf_norm(m1, tol=1e-4, n_jobs=-1):
    # Solve original model once
    m1.optimize()
    if m1.status != gp.GRB.OPTIMAL:
        raise RuntimeError("Original model is not optimal.")

    sol_star = np.array([v.X for v in m1.getVars()])

    # --- Free pre-filter: skip constraints with large slack (clearly non-binding) ---
    C = list(m1.getConstrs())
    candidates = []
    for constr in C:
        slack = constr.Slack
        # Only test constraints that are near-active (slack ≈ 0)
        if abs(slack) <= tol * 10:
            candidates.append(constr)
    
    print(f"Total constraints: {len(C)}, candidates to test: {len(candidates)}")

    # --- Parallel testing of candidate constraints ---
    # We pass the model as a file to avoid pickling Gurobi objects
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".mps", delete=False) as f:
        tmp_path = f.name
    m1.write(tmp_path)

    # Get constraint names for candidates
    candidate_names = [c.ConstrName for c in candidates]
    sol_star_copy = sol_star.copy()

    def test_constraint(constr_name):
        import gurobipy as gp_local
        import numpy as np_local

        env = gp_local.Env()
        env.setParam("OutputFlag", 0)  # suppress output
        env.setParam("LogToConsole", 0)

        m_copy = gp_local.read(tmp_path, env)
        m_copy.setParam("OutputFlag", 0)

        c = m_copy.getConstrByName(constr_name)
        if c is None:
            return constr_name, False
        m_copy.remove(c)
        m_copy.update()
        m_copy.optimize()

        if m_copy.status == gp_local.GRB.OPTIMAL:
            sol_new = np_local.array([v.X for v in m_copy.getVars()])
            diff = np_local.linalg.norm(sol_star_copy - sol_new, ord=np_local.inf)
            return constr_name, diff > tol
        return constr_name, False  # infeasible/unbounded → not necessary per your logic

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=2)(
        delayed(test_constraint)(name) for name in candidate_names
    )

    os.unlink(tmp_path)  # clean up temp file

    # Collect necessary constraints
    necessary_names = {name for name, is_necessary in results if is_necessary}
    necessary_constraints = [c for c in C if c.ConstrName in necessary_names]

    return necessary_constraints

# Usage
s_N = len(prune_constraints_inf_norm(m1, tol=1e-4))
print(f"Number of support constraints: {s_N}")

Total constraints: 147601, candidates to test: 3


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.


Set parameter Username
Set parameter Username
Set parameter Username
Set parameter LicenseID to value 2778218
Set parameter LicenseID to value 2778218
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Academic license - for non-commercial use only - expires 2027-02-11
Number of support constraints: 3


[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    4.5s finished


#### nonconvex SOP PAC bound
Consider Equation (7) in https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=8299432

#### comparison of $\beta,\alpha$, and $\mathcal{N}$.

In [22]:
# as alternative, consider eqn 7 in the paper
def epsil(k,N1,beta1):
    res = (beta1/(N1*comb(N1, k)))**(1/(N1-k))
    return 1-res

In [23]:
beta = 10**-6
alpha_sN = epsil(s_N, N, beta)

print(f"With confidence of {(1-beta)*100:.6f}%, the non-violation of at least {1-alpha_sN:.6f}, and violation {alpha_sN:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.990751, and violation 0.009249


In [ ]:
# t_init = time.time()
# # ASF 2 Enforce sum2 - Asf <= vartheta

# def compute_sys_dyn(l, i):
#     return l, i, sys_dyn_hat(X_pairs[l][1], U[i]), sys_dyn(X_pairs[l][0], U[i])

# # Step 1: Compute system dynamics in parallel
# num_jobs = -1  # Use all available cores
# dyn_results = Parallel(n_jobs=num_jobs, verbose=2)(
#     delayed(compute_sys_dyn)(l, i) for l in range(len(X_pairs)) for i in range(len(U))
# )

# # Step 2: Add constraints sequentially (Gurobi constraint addition must be sequential during parallelization)
# for l, i, dyn_hat_output, dyn_output in dyn_results:
#     m1.addConstr(
#         gp.quicksum(1/(len(dyn_hat_output)) * asf_value(dyn_output, yn) for yn in dyn_hat_output) # sigma taken uniformly
#         - tau_b*asf_value(X_pairs[l][0], X_pairs[l][1]) + (tau_b-1)*delta_b <= vartheta
#     )

# t_fin = time.time()
# diff_t = t_fin - t_init
# m1.update()
# print(diff_t)